# TDL-ADD — Aşama 2: Mamba Eğitimi (Colab)
**Hücre çalıştırma sırası:** 1 → 2 → 3 → 4  
Hücre 3 uzun sürer (~30–60 dk preprocess). Runtime yeniden başlatılırsa sadece Hücre 4'ü çalıştırabilirsiniz (özellikler `/content/` diskinde duruyorsa).

In [ ]:
# ── Hücre 1: Drive Bağlama + Repo Klonlama ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Checkpoint ve score çıktıları için Drive klasörlerini önceden oluştur
import os
os.makedirs('/content/drive/MyDrive/TDL-ADD/models/A2_Mamba/checkpoints_A2_Mamba', exist_ok=True)
os.makedirs('/content/drive/MyDrive/TDL-ADD/scores/A2_Mamba', exist_ok=True)

# Repo klonla (zaten varsa atla)
if not os.path.exists('/content/TDL-ADD'):
    !git clone https://github.com/CenkAydin/TDL-ADD /content/TDL-ADD
else:
    print('Repo zaten mevcut, güncelleniyor...')
    !git -C /content/TDL-ADD pull

%cd /content/TDL-ADD
print('Çalışma dizini:', os.getcwd())

In [ ]:
# ── Hücre 2: Kütüphane Kurulumları ────────────────────────────────────────────
# CUDA versiyonunu kontrol et — mamba-ssm CUDA 11.8+ gerektirir
!nvcc --version
!python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda)"

# causal-conv1d önce kurulmalı (mamba-ssm'nin zorunlu bağımlılığı)
!pip install causal-conv1d>=1.4.0 --quiet
!pip install mamba-ssm --quiet

# Kurulumu doğrula
!python -c "from mamba_ssm import Mamba; print('mamba-ssm başarıyla import edildi.')"

# Diğer gereksinimler
!pip install transformers tqdm pytorch-model-summary --quiet

In [ ]:
# ── Hücre 3: Veri İndirme + Açma + Preprocess ─────────────────────────────────
# Bu hücre ~30–60 dk sürebilir. Zenodo arşivi ~25 GB'dır.
import os

DATA_ROOT    = '/content/asv2019PS/database'
FEATURE_ROOT = '/content/asv2019PS/preprocess_A1_WavLM_Large'
ARCHIVE_PATH = '/content/asv2019ps_archive.zip'

os.makedirs(DATA_ROOT, exist_ok=True)

# 1) Zenodo arşivini indir (zaten varsa atla)
if not os.path.exists(ARCHIVE_PATH):
    print('Zenodo arşivi indiriliyor (~25 GB)...')
    !wget -q --show-progress \
        'https://zenodo.org/api/records/5766198/files-archive' \
        -O {ARCHIVE_PATH}
else:
    print('Arşiv zaten mevcut, indirme atlandı.')

# 2) Arşivi aç
if not os.path.exists(os.path.join(DATA_ROOT, 'train')):
    print('Arşiv açılıyor...')
    !unzip -q {ARCHIVE_PATH} -d /content/asv2019PS_raw
    # Açılan klasör yapısını incele ve doğru konuma taşı
    !ls /content/asv2019PS_raw/
    # NOT: Arşiv içindeki klasör adına göre aşağıdaki komutu güncelle:
    # !mv /content/asv2019PS_raw/<KLASOR_ADI>/* {DATA_ROOT}/
else:
    print('Veri zaten açılmış.')

# 3) WavLM-Large ile özellik çıkartma
if not os.path.exists(os.path.join(FEATURE_ROOT, 'train', 'wavlm-large')):
    print('Özellik çıkartma başlıyor...')
    !python /content/TDL-ADD/preprocess.py \
        --database_dir {DATA_ROOT} \
        --protocol_dir /content/TDL-ADD/label \
        --output_dir   {FEATURE_ROOT}
else:
    print('Özellikler zaten çıkartılmış.')

In [ ]:
# ── Hücre 4: Eğitimi Başlat ───────────────────────────────────────────────────
# Çıktılar: checkpointler ve loglar doğrudan Drive'a yazılır.
!python /content/TDL-ADD/main_train.py \
    -m  TDL_Mamba \
    -f  /content/asv2019PS/preprocess_A1_WavLM_Large \
    -d  /content/asv2019PS/database \
    -o  /content/drive/MyDrive/TDL-ADD/models/A2_Mamba \
    --ckpt_subdir checkpoints_A2_Mamba \
    --num_epochs  200 \
    --batch_size  24 \
    --lr          0.0001 \
    --lam         0.1 \
    --num_workers 2 \
    --base_loss   bce \
    --gpu         0